# Closed-Form Linear Regression and Regularization

This notebook compares the closed-form solution with gradient descent and introduces L2 regularization.

We use the same normalized squared-error objective as the previous notebooks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
y = np.array([2.2, 3.9, 6.1, 7.8, 10.2])
X_aug = np.c_[np.ones(len(X)), X]
print(f'Training data: {len(X)} observations')

## 1. Closed-form solution

For the unregularized least-squares problem,

$$
J(\theta)=\frac{1}{2n}\lVert X\theta-y\rVert^2,
$$

setting the gradient to zero gives the normal equations

$$
X^T X\theta=X^T y.
$$

We solve this linear system directly rather than explicitly computing a matrix inverse.

In [ ]:
theta_closed = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)
print('Closed-form theta:', theta_closed)

## 2. Compare with gradient descent

The gradient of the same objective is

$$
\nabla J(\theta)=\frac{1}{n}X^T(X\theta-y).
$$

Using the same normalization here is important: gradient descent should optimize exactly the objective defined above.

In [ ]:
theta_gd = np.zeros(X_aug.shape[1])
learning_rate = 0.05

for _ in range(2000):
    errors = X_aug @ theta_gd - y
    gradient = X_aug.T @ errors / len(y)
    theta_gd -= learning_rate * gradient

print('Gradient-descent theta:', theta_gd)
print('Difference:', theta_gd - theta_closed)

## 3. L2 regularization

We now add an L2 penalty while leaving the intercept unpenalized. Using the same $1/(2n)$ scaling as the data-fit term, define

$$
J_{\mathrm{reg}}(\theta)=\frac{1}{2n}\lVert X\theta-y\rVert^2+\frac{\lambda}{2}\lVert\theta_{1:}\rVert^2.
$$

The corresponding normal equations are

$$
(X^TX+n\lambda I)\theta=X^Ty,
$$

where the intercept entry of $I$ is set to zero so it is not regularized.

In [ ]:
lambda_ = 1.0
penalty = len(y) * lambda_ * np.eye(X_aug.shape[1])
penalty[0, 0] = 0.0
theta_ridge = np.linalg.solve(
    X_aug.T @ X_aug + penalty,
    X_aug.T @ y,
)
print('Regularized theta:', theta_ridge)

## 4. Visual comparison

The regularized line is pulled toward smaller slope magnitude. This illustrates the trade-off introduced by the L2 penalty.

In [ ]:
xx = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
XX = np.c_[np.ones(len(xx)), xx]

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], y, s=70, label='training data')
plt.plot(xx[:, 0], XX @ theta_closed, linewidth=2, label='closed form')
plt.plot(xx[:, 0], XX @ theta_ridge, linewidth=2, label='L2 regularized')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Closed-Form vs. Regularized Linear Regression')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()